# Matriks Ablasi Tugas 2 vs Tugas 3 yang Dapat Direproduksi (dengan CI)

Skor dihitung ulang dari prediksi tersimpan:
- Model A (`LR/HGB [Harga]`) dan B_1 (`[Harga+NLP]`, korpus penuh) dari `data/results/predictions.csv`;
- Model B_2 (`[Harga+NLP]`, korpus terfilter) dari `data/results_filtered/predictions.csv`;
- Model C (`C2_finbert_art_cls_HGB`) dari `results_c/preds/C2_finbert_art_cls_HGB_test.csv` dan skor validation di `results_c/`.

Lalu dicek bahwa hasilnya identik dengan `results_c/ablation_matrix.csv` (buatan `src/06_ablation.ipynb`), dan ditambah
CI 95% *moving-block bootstrap* (blok 5, 2.000 ulangan, seed 42) untuk DA, F1-macro, MCC, AUC serta selisih berpasangan terhadap Model A.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../src").resolve()))
from c_common import *          # metrics(), SEED, DATA, RESULTS_C (path relatif ke ../data)
import os
os.chdir(Path("../src").resolve())   
ANALYSIS = RESULTS_C / "analysis"
ANALYSIS.mkdir(exist_ok=True)

In [2]:
full = pd.read_csv(DATA / "results" / "predictions.csv", parse_dates=["date"])
filt = pd.read_csv(DATA / "results_filtered" / "predictions.csv", parse_dates=["date"])
c_test = pd.read_csv(RESULTS_C / "preds" / "C2_finbert_art_cls_HGB_test.csv", parse_dates=["date"])
c_rec = json.loads((RESULTS_C / "C2_finbert_art_cls_HGB.json").read_text())

SUMBER = {  # nama di ablation_matrix.csv -> (file prediksi, kolom model)
    "A LR": (full, "LR [Harga]"), "A HGB": (full, "HGB [Harga]"),
    "B_1 LR": (full, "LR [Harga+NLP]"), "B_1 HGB": (full, "HGB [Harga+NLP]"),
    "B_2 LR": (filt, "LR [Harga+NLP]"), "B_2 HGB": (filt, "HGB [Harga+NLP]"),
}
baris, pred_test = [], {}
for nama, (P, kol) in SUMBER.items():
    r = {"name": nama}
    for sp in ["val", "test"]:
        q = P[P.split == sp]
        r.update({f"{sp}_{k}": v for k, v in metrics(q.target, q[f"pred::{kol}"], q[f"score::{kol}"]).items()})
    q = P[P.split == "test"]
    pred_test[nama] = (q["target"].values, q[f"pred::{kol}"].values, q[f"score::{kol}"].values)
    baris.append(r)
assert (c_test["date"].values == full[full.split == "test"]["date"].values).all()
yc, sc = c_test["target"].values, c_test["score"].values
pred_test["C"] = (yc, (sc >= 0.5).astype(int), sc)
baris.append({"name": "C (FinBERT art-CLS)", **{f"val_{k}": v for k, v in c_rec["val"].items()},
              **{f"test_{k}": v for k, v in metrics(yc, (sc >= 0.5).astype(int), sc).items()}})
M_ = pd.DataFrame(baris)
M_.round(3)

,name,val_DA,val_BalAcc,val_F1_macro,val_MCC,val_AUC,test_DA,test_BalAcc,test_F1_macro,test_MCC,test_AUC
0,A LR,0.506,0.508,0.504,0.016,0.501,0.517,0.515,0.513,0.031,0.527
1,A HGB,0.523,0.521,0.520,0.042,0.503,0.500,0.503,0.498,0.005,0.538
2,B_1 LR,0.517,0.516,0.516,0.032,0.474,0.517,0.517,0.514,0.034,0.539
3,B_1 HGB,0.517,0.516,0.515,0.031,0.471,0.455,0.457,0.453,-0.086,0.491
4,B_2 LR,0.528,0.531,0.527,0.062,0.529,0.500,0.505,0.499,0.009,0.561
5,B_2 HGB,0.591,0.592,0.591,0.184,0.592,0.506,0.518,0.506,0.037,0.507
6,C (FinBERT art-CLS),0.591,0.588,0.588,0.179,0.568,0.584,0.561,0.560,0.127,0.594


## Cek: sama dengan `results_c/ablation_matrix.csv`?

In [3]:
ref = pd.read_csv(RESULTS_C / "ablation_matrix.csv")
kol = [c for c in ref.columns if c.startswith(("val_", "test_")) and c != "test_auc_ci95"]
selisih = (M_.set_index("name")[kol] - ref.set_index("name")[kol]).abs()
print("selisih maksimum:", float(selisih.max().max()))
print("IDENTIK" if selisih.max().max() < 1e-12 else "ADA SELISIH, cek tabel di bawah")
selisih.max(axis=1)

selisih maksimum: 8.326672684688674e-17
IDENTIK


name
A LR                   4.510281e-17
A HGB                  6.158268e-17
B_1 LR                 6.245005e-17
B_1 HGB                5.551115e-17
B_2 LR                 4.857226e-17
B_2 HGB                5.551115e-17
C (FinBERT art-CLS)    8.326673e-17
dtype: float64

## CI 95% (moving-block bootstrap) di test

In [5]:
def mbb_index(n, B=2000, block=5, seed=SEED):
    g = np.random.default_rng(seed)
    nb = int(np.ceil(n / block))
    for _ in range(B):
        yield (g.integers(0, n - block + 1, nb)[:, None] + np.arange(block)).ravel()[:n]

n = len(yc)
IDX = list(mbb_index(n))     # indeks bootstrap yang sama untuk semua model -> selisih berpasangan
def ci(vals):
    return [round(float(v), 3) for v in np.nanpercentile(vals, [2.5, 97.5])]

hasil = []
yA, pA, sA = pred_test["A HGB"]
for nama, (y_, p_, s_) in pred_test.items():
    boot = {k: [] for k in ["DA", "F1_macro", "MCC", "AUC", "dDA_vs_A_HGB", "dAUC_vs_A_HGB"]}
    for ix in IDX:
        m = metrics(y_[ix], p_[ix], s_[ix]) if len(set(y_[ix])) == 2 else None
        if m is None:
            continue
        mA = metrics(yA[ix], pA[ix], sA[ix])
        for k in ["DA", "F1_macro", "MCC", "AUC"]:
            boot[k].append(m[k])
        boot["dDA_vs_A_HGB"].append(m["DA"] - mA["DA"])
        boot["dAUC_vs_A_HGB"].append(m["AUC"] - mA["AUC"])
    titik = metrics(y_, p_, s_)
    hasil.append({"name": nama, **{f"test_{k}": round(titik[k], 3) for k in ["DA", "F1_macro", "MCC", "AUC"]},
                  **{f"ci_{k}": ci(v) for k, v in boot.items()}})
CI = pd.DataFrame(hasil)
pd.set_option("display.max_columns", 20); pd.set_option("display.width", 200)
CI

,name,test_DA,test_F1_macro,test_MCC,test_AUC,ci_DA,ci_F1_macro,ci_MCC,ci_AUC,ci_dDA_vs_A_HGB,ci_dAUC_vs_A_HGB
0,A LR,0.517,0.513,0.031,0.527,"[0.455, 0.59]","[0.449, 0.588]","[-0.091, 0.181]","[0.453, 0.615]","[-0.051, 0.101]","[-0.082, 0.084]"
1,A HGB,0.500,0.498,0.005,0.538,"[0.421, 0.573]","[0.417, 0.567]","[-0.152, 0.148]","[0.445, 0.624]","[0.0, 0.0]","[0.0, 0.0]"
2,B_1 LR,0.517,0.514,0.034,0.539,"[0.444, 0.596]","[0.441, 0.594]","[-0.11, 0.193]","[0.456, 0.624]","[-0.056, 0.107]","[-0.079, 0.098]"
3,B_1 HGB,0.455,0.453,-0.086,0.491,"[0.388, 0.534]","[0.383, 0.533]","[-0.221, 0.08]","[0.415, 0.576]","[-0.124, 0.056]","[-0.123, 0.05]"
4,B_2 LR,0.500,0.499,0.009,0.561,"[0.427, 0.573]","[0.425, 0.573]","[-0.131, 0.158]","[0.489, 0.646]","[-0.09, 0.096]","[-0.07, 0.133]"
5,B_2 HGB,0.506,0.506,0.037,0.507,"[0.438, 0.573]","[0.437, 0.573]","[-0.098, 0.177]","[0.42, 0.589]","[-0.079, 0.09]","[-0.106, 0.051]"
6,C,0.584,0.560,0.127,0.594,"[0.506, 0.652]","[0.478, 0.629]","[-0.037, 0.268]","[0.519, 0.666]","[-0.006, 0.18]","[-0.053, 0.165]"


In [6]:
CI.to_csv(ANALYSIS / "ablasi_ci_bootstrap.csv", index=False)
print("tersimpan:", ANALYSIS / "ablasi_ci_bootstrap.csv")

tersimpan: ..\results_c\analysis\ablasi_ci_bootstrap.csv
